In [29]:
import pandas as pd
import json
import re
import unicodedata 
# Your existing 6-entity CSV
df = pd.read_csv("NER_output_6_entities.csv")

# Original ER_NER JSON
with open("dataset_with_codes.json", "r", encoding="utf-8") as f:
    data = json.load(f)

allergy_target = {}
for record in data:
    doc_id = record["doc_id"]
    target = 0  # default = negative

    for annotation in record.get("annotations", []):
        if annotation.get("label") == "Alergias medicamentosas":

            if annotation.get("Polaridade") == "Positiva":
                target = 1
                break

            elif annotation.get("Polaridade") == "Negativa":
                target = 0

    allergy_target[doc_id] = target

# Add target column to your 6-entity dataset
df["medication_allergy"] = df["doc_id"].map(allergy_target)

# Save new dataset
df.to_csv(
    "NER_output_6_entities_with_medication_allergy.csv",
    index=False,
    encoding="utf-8"
)

# Check result
print(df["medication_allergy"].value_counts())
print(df.head(3))

medication_allergy
0    162
1    138
Name: count, dtype: int64
   doc_id                                            symptom  \
0       1  dor abdominal intensa; vómitos; dor abdominal;...   
1       2  Dor abdominal; dor abdominal intensa; edema; h...   
2       3                                                NaN   

                                          medication  \
0  Insulina Novomix 60 U ao pequeno-almoço + 40 U...   
1  Metformina 1000mg bid; Liraglutido 1.8 (Victoz...   
2  Enalapril 20mg id; Amlodipino 10mg id; Simvast...   

                           treatment diagnosis  \
0  regime alimentar; insulinotratado       NaN   
1                     Antibioterapia       NaN   
2                                NaN       NaN   

                                           condition  \
0  doença cardiovascular; diabetes tipo 2; Cirros...   
1                               dislipidemia; anemia   
2  doenca pulmonar; Doenca pulmonar; Hipertensao ...   

                             

In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import joblib
from sklearn.model_selection import (train_test_split, GridSearchCV,StratifiedKFold, cross_val_predict) 
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC                                         
from sklearn.pipeline import Pipeline                                      
from sklearn.compose import ColumnTransformer         
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, f1_score, average_precision_score, precision_recall_curve)  
from sklearn.feature_extraction.text import TfidfVectorizer
print(df.shape)
df = df[['doc_id', 'symptom', 'medication', 'treatment', 'diagnosis',
         'condition', 'exam', 'medication_allergy']].copy()
print("\nColumns:")
print(df.columns.tolist())
print(df.isnull().sum())


(300, 8)

Columns:
['doc_id', 'symptom', 'medication', 'treatment', 'diagnosis', 'condition', 'exam', 'medication_allergy']
doc_id                  0
symptom                11
medication             17
treatment             198
diagnosis             171
condition              20
exam                   32
medication_allergy      0
dtype: int64


In [31]:
print("rows:", len(df), "| exact duplicate rows:", df.duplicated().sum())
text_columns = ["symptom","medication", "treatment","diagnosis","condition", "exam"]

for col in text_columns:
    df[col] = df[col].fillna("").astype(str)

print(df.isnull().sum())

df["combined_text"] = df[text_columns].agg( " ".join,
    axis=1
)
print(df['combined_text'])


rows: 300 | exact duplicate rows: 0
doc_id                0
symptom               0
medication            0
treatment             0
diagnosis             0
condition             0
exam                  0
medication_allergy    0
dtype: int64
0      dor abdominal intensa; vómitos; dor abdominal;...
1      Dor abdominal; dor abdominal intensa; edema; h...
2       Enalapril 20mg id; Amlodipino 10mg id; Simvas...
3      dispneia; dor torácica; febre; tosse; edema lo...
4      tosse; febre; dor torácica; edema Salbutamol q...
                             ...                        
295    dor abdominal; diarreia; náuseas; vómitos; feb...
296    edema Prednisolona 20mg manhã; MMF 500mg bid; ...
297    Prurido; dispneia; dor torácica; tosse; Edema;...
298    Edema; edema; fadiga; falta de ar; febre; conf...
299    febre; tosse salbutamol); lisinopril Oxigenote...
Name: combined_text, Length: 300, dtype: str


In [32]:
X = df["combined_text"]

y = df["medication_allergy"]


print("\nTarget distribution:")
print(y.value_counts())




Target distribution:
medication_allergy
0    162
1    138
Name: count, dtype: int64


In [33]:
#  ****TRAINING CLASSIFICATION(LOGISTIC REGRESSION MODEL)****
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print("\nTraining Data:", len(X_train))
print("Testing Data:", len(X_test))



Training Data: 240
Testing Data: 60


In [34]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)


print("\nTF-IDF training shape:")
print(X_train_tfidf.shape)

print("TF-IDF testing shape:")
print(X_test_tfidf.shape)


TF-IDF training shape:
(240, 3179)
TF-IDF testing shape:
(60, 3179)


In [ ]:
model = LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42)
model.fit(X_train_tfidf,y_train)

y_pred = model.predict(X_test_tfidf)
y_probability = model.predict_proba(
    X_test_tfidf
)[:, 1]
a1

In [36]:
accuracy = accuracy_score(y_test,y_pred)
roc_auc = roc_auc_score(y_test, y_probability)
print("LOGISTIC REGRESSION RESULTS")
print("\nAccuracy:")
print(round(accuracy, 4))
print("\nROC-AUC:")
print(round(roc_auc, 4))
print("\nClassification Report:")
print(classification_report( y_test,y_pred,target_names=[  "No Medication Allergy", "Medication Allergy" ] ))


print("\nConfusion Matrix:")
print(confusion_matrix( y_test, y_pred ))
results = pd.DataFrame({
    "text": X_test.values,
    "actual": y_test.values,
    "predicted": y_pred,
    "probability_medication_allergy": y_probability})
print("\nSample predictions:")
print(results.head(10))
results.to_csv( "logistic_regression_medication_allergy_predictions.csv",index=False,encoding="utf-8")
print("\nPredictions saved to: " "logistic_regression_medication_allergy_predictions.csv")

LOGISTIC REGRESSION RESULTS



Accuracy:
0.6167

ROC-AUC:
0.5931

Classification Report:
                       precision    recall  f1-score   support

No Medication Allergy       0.67      0.69      0.68        35
   Medication Allergy       0.54      0.52      0.53        25

             accuracy                           0.62        60
            macro avg       0.60      0.60      0.60        60
         weighted avg       0.61      0.62      0.62        60


Confusion Matrix:
[[24 11]
 [12 13]]

Sample predictions:
                                                text  actual  predicted  \
0  tosse; fadiga._refere; dispneia; febre; fadiga...       0          0   
1  edema Ácido acetilsalicílico 100mg id; Metopro...       0          0   
2  edema Prednisolona 20mg manhã; Capecitabina 10...       0          0   
3  febre; perda de apetite; náuseas; tosse; edema...       0          0   
4  tosse; edema Salbutamol; Fluticasona; Salmeter...       1          0   
5  dispneia; tosse; febre; dor torácica; edema Si.